# Example 11.3.2 / 4: Control Substitution with substout

Companion notebook for **11.3.2 Substitution and Expression Expansion** in the AMPL lab Documentation, section "Control Substitution with substout". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Control Substitution with substout

Eligibility is concrete too. Declaring `var Cost >= 0;` gives the left-hand variable a bound, so
`CostDef` is no longer a defining constraint and `substout 1` leaves two variables and one
constraint:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
var Ship >= 2, <= 10, := 3;  # shipment, bounded and started at 3
var Cost >= 0;               # bound makes CostDef ineligible for substout

minimize Total: Cost * Ship;  # total cost, unit cost times shipment

subject to CostDef: Cost = (10 + 4 * Ship) / (1 + Ship);  # kept as a constraint

Writing model.mod


### amplpy: run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# run.py
ampl.option["substout"] = 1
ampl.solve(solver="ipopt", verbose=False)
print(ampl.get_value("_snvars"), ampl.get_value("_sncons"))

2 1
